# 01 - Data Exploration

In [ ]:
import osos.chdir(r'C:\Users\avav\Documents\freqtrade')import pandas as pdimport plotly.graph_objects as gofrom plotly.subplots import make_subplotsimport numpy as npDATA = r'C:\Users\avav\Documents\freqtrade\user_data\data\bybit\futures'

In [ ]:
# Load all SOL futures timeframesframes = {}for tf in ['1m-futures', '5m-futures', '15m-futures', '1h-futures', '4h-futures', '1h-mark', '1h-funding_rate']:    p = f'{DATA}/SOL_USDT_USDT-{tf}.feather'    df = pd.read_feather(p)    if 'date' in df.columns and df['date'].dtype.kind != 'M':        df['date'] = pd.to_datetime(df['date'], unit='ms', utc=True)    frames[tf] = df    print(f"{tf:>20s}: {len(df):>9} rows, {df['date'].iloc[0]} -> {df['date'].iloc[-1]}")

In [ ]:
# Plot OHLCV for each timeframe in a single figurefig = make_subplots(rows=len(frames), cols=1, shared_xaxes=False,                    subplot_titles=list(frames.keys()), vertical_spacing=0.02)for i, (tf, df) in enumerate(frames.items(), 1):    fig.add_trace(go.Candlestick(        x=df['date'], open=df['open'], high=df['high'],        low=df['low'], close=df['close'], name=tf, showlegend=False    ), row=i, col=1)    fig.update_yaxes(title_text=tf, row=i, col=1)fig.update_layout(height=300*len(frames), template='plotly_dark', title='SOL/USDT:USDT OHLCV')fig.show()

In [ ]:
# Funding rate distributionfr = frames['1h-funding_rate']print("Funding rate stats:")print(fr['close'].describe())print(f"\nPositive rate: {(fr['close']>0).sum()} ({(fr['close']>0).mean():.1%})")print(f"Negative rate: {(fr['close']<0).sum()} ({(fr['close']<0).mean():.1%})")fig = go.Figure()fig.add_trace(go.Histogram(x=fr['close']*10000, nbinsx=80, name='Funding rate (bps)'))fig.update_layout(title='Funding rate distribution (bps)', template='plotly_dark', xaxis_title='bps', yaxis_title='count')fig.show()

In [ ]:
# Mark vs close divergencemark = frames['1h-mark'][['date', 'close']].rename(columns={'close': 'mark'})fut1h = frames['1h-futures'][['date', 'close']].rename(columns={'close': 'fut'})m = mark.merge(fut1h, on='date')m['basis_pct'] = (m['mark'] - m['fut']) / m['fut'] * 100print(m['basis_pct'].describe())fig = go.Figure()fig.add_trace(go.Scatter(x=m['date'], y=m['basis_pct'], mode='markers', name='basis_pct'))fig.add_hline(y=0, line_dash='dash', line_color='gray')fig.update_layout(title='Mark-future basis (1h, %)', template='plotly_dark')fig.show()

In [ ]:
# Data sanity: gaps in 1m datadf = frames['1m-futures']diffs = df['date'].diff().dropna()print("Most common 1m diff:", diffs.mode().iloc[0])print("Largest 1m gap:", diffs.max())top_gaps = diffs.sort_values(ascending=False).head(20)print("\nTop 20 gaps:")print(top_gaps)

In [ ]:
Data exploration complete.